# Importing and cleaning datasets for final visualizations

In [ ]:
import pandas as pd
import re
import pycountry
import matplotlib.pyplot as plt

import pandas as pd
import json
import re
import os

## "Dwarfing" Fish Production/Landings Graph 

Data Source: FIRMS Global Tuna Atlas (https://www.fao.org/fishery/en/collection/firms-tuna-atlas)

Acronym: GTA

metadata here: https://www.fao.org/fishery/geonetwork/srv/eng/catalog.search#/metadata/global_nominal_catch_firms_level0



Catches of Bluefin species + Bigeye and Yellofin tuna up to 2025

In [ ]:
#import cell is seperate so reach running of notebook starts from the same place (data un-altered)
GTA_all = pd.read_csv("../datasets/global_nominal_catch_firms_level0_harmonized.csv")
GTA_all.head()


In [ ]:
GTA_all.dtypes
GTA_all["measurement_type"].value_counts()


In [ ]:
#first four numbers in time_start is the year -> extract and make new column
GTA_all['year'] = GTA_all['time_start'].str[:4]
#remove:
    #gear_type - cannot reasonably assume all tuna caught with certain gear type are used for the same purpose (fishing vs ranching)
    #fishing_mode
    #measurement_processing_level
    #measurement_unit - the same for all (it's tonnes)
    #source_authority
    #geographic_identifier
    #measurement
GTA_all = GTA_all.drop(columns=["measurement",'gear_type', 'fishing_mode', 'measurement_processing_level', 'measurement_unit', 'source_authority', 'geographic_identifier'])
#remove rows with measurement_type DD or DL or NL (Nominal Landings, which is a subset of NC Nominal Catch)
GTA_all = GTA_all[~GTA_all['measurement_type'].isin(['DD', 'DL', 'NL'])]
#and then drop the measurement_type column since we are only looking at one type of measurement (NC)
GTA_all = GTA_all.drop(columns=['measurement_type'])
#drop time start, time end
GTA_all = GTA_all.drop(columns=['time_start', 'time_end'])


#keep the following fish: 
    #BFT Atlantic bluefin tuna
    #BET Bigeye tuna
    #PBF Pacific bluefin tuna
    #SBF Southern bluefin tuna
    #YFT Yellowfin tuna
tuna_keep = ['BFT', 'BET', 'PBF', 'SBF', 'YFT']


GTA_tuna = GTA_all[GTA_all['species'].isin(tuna_keep)]
GTA_tuna_grouped = GTA_tuna.groupby(['species', 'year'])['measurement_value'].sum().reset_index()


In [ ]:
GTA_tuna.head()

In [ ]:
GTA_tuna["fishing_fleet"].value_counts()
#if country has "EU" as first two letters in name, delete those letters and keep the rest as country name (e.g. "EUESP" -> "ESP")
GTA_tuna['fishing_fleet'] = GTA_tuna['fishing_fleet'].apply(lambda x: re.sub(r'^EU', '', x))

In [ ]:
GTA_tuna["fishing_fleet"].value_counts()

In [ ]:
#convertor from fishing_fleet to country name 
    #each fishing fleet is a three letter abbreviation of a country name
    #these are ISO 3166 codes, more info here https://www.iso.org/iso-3166-country-codes.html
def fleet_to_country(fleet_code):
    try:
        country = pycountry.countries.get(alpha_3=fleet_code)
        return country.name
    except:
        return None
    
GTA_tuna['country'] = GTA_tuna['fishing_fleet'].apply(fleet_to_country)



In [ ]:
GTA_tuna["species"].unique()

In [ ]:
#what rows in GTA_tuna have emtpy country column?
GTA_tuna[GTA_tuna['country'].isna()]['fishing_fleet'].unique()

### Export GTA FIRMS

In [ ]:
## EXPORT
GTA_tuna.to_csv("../datasets/export/GTA_FIRMs_tuna_cleaned_countries.csv", index=False)
GTA_tuna_grouped.to_csv("../datasets/export/GTA_FIRMs_tuna_cleaned_grouped.csv", index=False)


In [ ]:
GTA_tuna_grouped.head()

#total catch of SBF BFT PBF in 1965
GTA_tuna_grouped[(GTA_tuna_grouped['species'].isin(['SBF', 'BFT', 'PBF'])) & (GTA_tuna_grouped['year'] == '2007')]['measurement_value'].sum()

In [ ]:
GTA_tuna.head(10)
#only include SBF, BFT, PBF, group by country and year, total catch in 1965
GTA_tuna[(GTA_tuna['species'].isin(['SBF', 'BFT', 'PBF'])) & (GTA_tuna['year'] == '2007')].groupby(['year'])['measurement_value'].sum()

#total catch of SBF BFT PBF summed across all years
GTA_tuna[(GTA_tuna['species'].isin(['SBF', 'BFT', 'PBF']))]['measurement_value'].sum()

In [ ]:
#group by species and year

GTA_tuna_grouped.head(10)


#graph only bluefins, measurement_value over time
GTA_bluefins = GTA_tuna_grouped[GTA_tuna_grouped['species'].isin(['BFT', 'PBF', 'SBF'])]
plt.figure(figsize=(10,6))
for species in ['BFT', 'PBF', 'SBF']:
    species_data = GTA_bluefins[GTA_bluefins['species'] == species]
    plt.plot(species_data['year'], species_data['measurement_value'], label=species)
plt.xlabel('Year')
plt.ylabel('Measurement Value (tonnes)')
plt.title('Global Nominal Catch of Bluefin Tuna Over Time')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#same but stacked barplot
GTA_bluefins_pivot = GTA_bluefins.pivot(index='year', columns='species', values='measurement_value').fillna(0)
GTA_bluefins_pivot.plot(kind='bar', stacked=True, figsize=(10,6))
plt.xlabel('Year')
plt.ylabel('Measurement Value (tonnes)')
plt.title('Global Nominal Catch of Bluefin Tuna Over Time (Stacked)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#stacked barplot of all tuna species
GTA_tuna_pivot = GTA_tuna_grouped.pivot(index='year', columns='species', values='measurement_value').fillna(0)
GTA_tuna_pivot.plot(kind='bar', stacked=True, figsize=(10,6))
plt.xlabel('Year')
plt.ylabel('Measurement Value (tonnes)')
plt.title('Global Nominal Catch of Tuna Species Over Time (Stacked)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Map - 5° Lat-Long of Global Tuna Catches 

Source: GTA FIRMS 


will call this object "tuna_map"

In [ ]:
tuna_map = pd.read_csv("datasets/tuna_map/global_catch_5deg_1m_firms_level0_harmonized.csv")
tuna_map.head()

In [ ]:
tuna_map["measurement_unit"].value_counts()

In [ ]:
#this is a pretty big dataset that includes all tuna and shark like species, so will treat it similarly to the previous dataset.


#create "year" column from time_start
tuna_map['year'] = tuna_map['time_start'].str[:4]

#only retain species of interest
tuna_keep = ['BFT', 'BET', 'PBF', 'SBF', 'YFT']
tuna_map_soi = tuna_map[tuna_map['species'].isin(tuna_keep)]

#remove rows with measurement_type DD or DL
tuna_map_soi = tuna_map_soi[~tuna_map_soi['measurement_type'].isin(['DD', 'DL'])]

#remove:
    #gear_type - cannot reasonably assume all tuna caught with certain gear type are used for the same purpose (fishing vs ranching)
    #fishing_mode
    #measurement_processing_level
    #source_authority
    #measurement
    #time_start
    #time_end
tuna_map_soi = tuna_map_soi.drop(columns=['gear_type', 'fishing_mode', 'measurement_processing_level', 'source_authority', 'measurement', 'measurement_type', 'time_start', 'time_end'])



In [ ]:
tuna_map_soi.head()

In [ ]:
tuna_map_soi["measurement_unit"].value_counts()

In [ ]:
#map with only tuna of interst, only neccesary columns
tuna_map_soi.head()

#need to figure out what to do with count vs tonne data
tuna_map_soi_tonne = tuna_map_soi[tuna_map_soi['measurement_unit'] == 't']
tuna_map_soi_count = tuna_map_soi[tuna_map_soi['measurement_unit'] == 'no']


In [ ]:
#group by geographic identifier, year, and species, sum measurement value
tuna_map_soi_count_grouped = tuna_map_soi_count.groupby(['geographic_identifier', 'year', 'species'])['measurement_value'].sum().reset_index()
tuna_map_soi_tonne_grouped = tuna_map_soi_tonne.groupby(['geographic_identifier', 'year', 'species'])['measurement_value'].sum().reset_index()
tuna_map_soi_count_grouped.head()

In [ ]:
#what is the first year of data for each species?
tuna_map_soi_count_grouped.groupby('species')['year'].min()

#remove years before 1965 for even coverage across species
tuna_map_soi_count_grouped = tuna_map_soi_count_grouped[tuna_map_soi_count_grouped['year'] >= '1965']
tuna_map_soi_tonne_grouped = tuna_map_soi_tonne_grouped[tuna_map_soi_tonne_grouped['year'] >= '1965']
tuna_map_soi_tonne_grouped.head()

In [ ]:
#bluefin (BFT, PBF, SBF) only subset
tuna_map_soi_tonne_bluefin = tuna_map_soi_tonne_grouped[tuna_map_soi_tonne_grouped['species'].isin(['BFT', 'PBF', 'SBF'])]
tuna_map_soi_count_bluefin = tuna_map_soi_count_grouped[tuna_map_soi_count_grouped['species'].isin(['BFT', 'PBF', 'SBF'])]

#### EXPORT tuna dataset for map

In [ ]:
#export tonne and count
tuna_map_soi_count_grouped.to_csv("datasets/export/tuna_map_count.csv")
tuna_map_soi_tonne_grouped.to_csv("datasets/export/tuna_map_tonne.csv")

#export bluefin subsets
tuna_map_soi_count_bluefin.to_csv("datasets/export/tuna_map_count_bluefin.csv")
tuna_map_soi_tonne_bluefin.to_csv("datasets/export/tuna_map_tonne_bluefin.csv")

### Geographic Key for tuna map

Source: https://www.fao.org/fishery/geonetwork/srv/eng/catalog.search#/metadata/cwp-grid-map-5deg_x_5deg

Key for 5° by 5° grid overlaid on earth

In [ ]:
map_geographic_key = pd.read_csv("datasets/tuna_map/cwp-grid-map-5deg_x_5deg.csv")
map_geographic_key.head()

transforming this into mapbox-readable geojson

In [ ]:
"""
build_catch_geojson.py
----------------------
Merges tuna catch CSVs (count + tonne) into the CWP 5° grid GeoJSON
using wide format (one column per year).

Inputs:
  - cwp-grid-5deg.geojson       (corrected grid from previous step)
  - tuna_map_count.csv
  - tuna_map_tonne.csv

Output:
  - cwp-grid-5deg-catch.geojson

Usage:
  python build_catch_geojson.py
"""

import pandas as pd
import json
import os

# ── Paths (edit these if needed) ─────────────────────────────────────────────
GRID_GEOJSON  = 'datasets/tuna_map/cwp-grid-5deg-var3.geojson'
COUNT_CSV     = 'datasets/export/tuna_map_count_bluefin.csv'
TONNE_CSV     = 'datasets/export/tuna_map_tonne_bluefin.csv'
OUTPUT_GEOJSON = 'datasets/export/cwp-grid-5deg-catch-bluefin.geojson'

# ── 1. Load and pivot catch data to wide format ───────────────────────────────
print("Loading catch data...")
count_df = pd.read_csv(COUNT_CSV)
tonne_df = pd.read_csv(TONNE_CSV)

def to_wide(df, prefix):
    """Aggregate all species per square per year, then pivot to wide."""
    return (df
        .groupby(['geographic_identifier', 'year'])['measurement_value']
        .sum()
        .reset_index()
        .pivot(index='geographic_identifier', columns='year', values='measurement_value')
        .rename(columns=lambda y: f'{prefix}_{y}'))

count_wide = to_wide(count_df, 'count')
tonne_wide = to_wide(tonne_df, 'tonne')

catch_wide = count_wide.join(tonne_wide, how='outer')
catch_wide.index.name = 'CWP_CODE'
catch_wide = catch_wide.reset_index()

print(f"  Wide table: {catch_wide.shape[0]} squares × {catch_wide.shape[1]} columns")

# ── 2. Load GeoJSON ───────────────────────────────────────────────────────────
print(f"Loading {GRID_GEOJSON}...")
with open(GRID_GEOJSON) as f:
    geojson = json.load(f)

# ── 3. Merge catch data into GeoJSON properties ───────────────────────────────
print("Merging...")
catch_lookup = catch_wide.set_index('CWP_CODE').to_dict(orient='index')

matched, unmatched = 0, 0
for feature in geojson['features']:
    code = feature['properties']['CWP_CODE']
    if code in catch_lookup:
        for k, v in catch_lookup[code].items():
            feature['properties'][k] = (
                None if (isinstance(v, float) and pd.isna(v))
                else round(float(v), 3)
            )
        matched += 1
    else:
        unmatched += 1

print(f"  Matched: {matched} | No catch data: {unmatched}")

# ── 4. Write output ───────────────────────────────────────────────────────────
print(f"Writing {OUTPUT_GEOJSON}...")
with open(OUTPUT_GEOJSON, 'w') as f:
    json.dump(geojson, f)

size_mb = os.path.getsize(OUTPUT_GEOJSON) / 1e6
print(f"Done. Output: {OUTPUT_GEOJSON} ({size_mb:.1f} MB)")

### FIRMS Global Tuna Atlas 1° lat-long
https://zenodo.org/records/17707421

In [ ]:
tuna_1deg = pd.read_csv("../datasets/tuna_map/global_catch_1deg_1m_surface_firms_level0_harmonized.csv")
tuna_1deg.head()


In [ ]:
#same processing as 5deg

#create "year" column from time_start
tuna_1deg['year'] = tuna_1deg['time_start'].str[:4]

#only retain species of interest
tuna_keep = ['BFT', 'PBF', 'SBF',]
tuna_1deg_soi = tuna_1deg[tuna_1deg['species'].isin(tuna_keep)]

#remove rows with measurement_type DD or DL
tuna_1deg_soi = tuna_1deg_soi[~tuna_1deg_soi['measurement_type'].isin(['DD', 'DL'])]


#remove:
    #gear_type - cannot reasonably assume all tuna caught with certain gear type are used for the same purpose (fishing vs ranching)
    #fishing_mode
    #measurement_processing_level
    #source_authority
    #measurement
    #time_start
    #time_end
tuna_1deg_soi = tuna_1deg_soi.drop(columns=['gear_type', 'fishing_mode', 'measurement_processing_level', 'source_authority', 'measurement', 'measurement_type', 'time_start', 'time_end'])

#filter out number (no) - onlly 487 rows compared to 1100822 tonne rows
tuna_1deg_soi = tuna_1deg_soi[tuna_1deg_soi['measurement_unit'] == 't']

tuna_1deg_soi.head()

In [ ]:
#group by geographic identifier, year, and species, sum measurement value
tuna_1deg_soi_grouped = tuna_1deg_soi.groupby(['geographic_identifier', 'year', 'species'])['measurement_value'].sum().reset_index()


#check earliest years and filter to same baseline
tuna_1deg_soi_grouped.groupby('species')['year'].min()

#separate 1975-present baseline dataset
tuna_1deg_1975 = tuna_1deg_soi_grouped[tuna_1deg_soi_grouped['year'] >= '1975']



In [ ]:
tuna_1deg_soi["measurement_unit"].value_counts()

geojson script below:

In [ ]:
"""
build_catch_geojson_1deg.py
---------------------------
Converts the CWP 1°x1° grid CSV + tuna catch CSV into a merged GeoJSON
for use in Mapbox, using wide format (one column per year).

Inputs:
  - cwp-grid-map-1deg_x_1deg.csv   (CWP grid with WKT geometry)
  - tuna_1deg_soi_grouped.csv      (catch data: geographic_identifier, year, species, measurement_value)

Outputs:
  - cwp-grid-1deg.geojson          (full grid, all squares)
  - cwp-grid-1deg-sea.geojson      (sea-only squares, smaller file)
  - cwp-grid-1deg-catch.geojson    (sea-only squares + catch data merged in)

Usage:
  python build_catch_geojson_1deg.py

Note on file size:
  At 1°x1° resolution (64,800 squares) the full GeoJSON is ~26 MB.
  The sea-only + catch merged output is ~28 MB. For Mapbox GL JS,
  consider hosting via Mapbox Tileset API (converts to vector tiles)
  rather than loading as a raw GeoJSON source, which will be slow.
"""



# ── Paths (edit these if needed) ─────────────────────────────────────────────
GRID_CSV        = '../datasets/tuna_map/cwp-grid-map-1deg_x_1deg.csv'
CATCH_CSV       = '../datasets/export/bluefin-tuna_1deg_soi_grouped.csv'
OUTPUT_GRID     = '../datasets/export/geojson_export/bluefin-cwp-grid-1deg.geojson'
OUTPUT_SEA      = '../datasets/export/geojson_export/bluefin-cwp-grid-1deg-sea.geojson'
OUTPUT_CATCH    = '../datasets/export/geojson_export/bluefin-cwp-grid-1deg-catch.geojson'

# ── Step 1: Convert grid CSV → GeoJSON (fix WKT axis swap) ───────────────────
print("Building grid GeoJSON...")

def parse_wkt_multipolygon(wkt_str):
    """
    Parse WKT MULTIPOLYGON to GeoJSON coordinates.
    The source WKT uses (lat, lon) order — this swaps all pairs to (lon, lat)
    as required by GeoJSON / Mapbox.
    """
    s = wkt_str.strip()[len('MULTIPOLYGON '):].strip()[1:-1]
    polygons = []
    for match in re.finditer(r'\(\((.*?)\)\)', s, re.DOTALL):
        rings = []
        for ring in re.split(r'\)\s*,\s*\(', match.group(1)):
            coords = []
            for pair in ring.strip().split(','):
                parts = pair.strip().split()
                if len(parts) >= 2:
                    lat, lon = float(parts[0]), float(parts[1])
                    coords.append([lon, lat])  # swap to GeoJSON (lon, lat)
            rings.append(coords)
        polygons.append(rings)
    return polygons

df = pd.read_csv(GRID_CSV)
features = []
for _, row in df.iterrows():
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "MultiPolygon",
            "coordinates": parse_wkt_multipolygon(row['the_geom'])
        },
        "properties": {
            "FID":       row['FID'],
            "CWP_CODE":  int(row['CWP_CODE']),
            "GRIDTYPE":  row['GRIDTYPE'],
            "QUADRANT":  row['QUADRANT'],
            "X_COORD":   row['X_COORD'],
            "Y_COORD":   row['Y_COORD'],
            "CWP_A":     row['CWP_A'],
            "CWP_B":     row['CWP_B'],
            "CWP_C":     row['CWP_C'],
            "CWP_D":     row['CWP_D'],
            "ON_SEA_P":  row['ON_SEA_P'],
            "ON_LAND_P": row['ON_LAND_P'],
        }
    })

geojson = {"type": "FeatureCollection", "features": features}
with open(OUTPUT_GRID, 'w') as f:
    json.dump(geojson, f)
print(f"  {len(features)} features → {os.path.getsize(OUTPUT_GRID)/1e6:.1f} MB  ({OUTPUT_GRID})")

# ── Step 2: Filter to sea-only squares ───────────────────────────────────────
print("Filtering to sea-only squares...")
sea_features = [f for f in features if (f['properties']['ON_SEA_P'] or 0) > 0]
sea_geojson = {"type": "FeatureCollection", "features": sea_features}
with open(OUTPUT_SEA, 'w') as f:
    json.dump(sea_geojson, f)
print(f"  {len(sea_features)} features → {os.path.getsize(OUTPUT_SEA)/1e6:.1f} MB  ({OUTPUT_SEA})")

# ── Step 3: Pivot catch data to wide format ───────────────────────────────────
print("Loading and pivoting catch data...")
catch_df = pd.read_csv(CATCH_CSV)

tonne_wide = (catch_df
    .groupby(['geographic_identifier', 'year'])['measurement_value']
    .sum()
    .reset_index()
    .pivot(index='geographic_identifier', columns='year', values='measurement_value')
    .rename(columns=lambda y: f'tonne_{y}'))
tonne_wide.index.name = 'CWP_CODE'
tonne_wide = tonne_wide.reset_index()

print(f"  {tonne_wide.shape[0]} squares × {tonne_wide.shape[1]} columns "
      f"(tonne_{catch_df['year'].min()} … tonne_{catch_df['year'].max()})")

# ── Step 4: Merge catch into sea GeoJSON ─────────────────────────────────────
print("Merging catch data into GeoJSON...")
catch_lookup = tonne_wide.set_index('CWP_CODE').to_dict(orient='index')

matched, unmatched = 0, 0
for feature in sea_geojson['features']:
    code = feature['properties']['CWP_CODE']
    if code in catch_lookup:
        for k, v in catch_lookup[code].items():
            feature['properties'][k] = (
                None if (isinstance(v, float) and pd.isna(v))
                else round(float(v), 3)
            )
        matched += 1
    else:
        unmatched += 1

print(f"  Matched: {matched} | No catch data: {unmatched}")

with open(OUTPUT_CATCH, 'w') as f:
    json.dump(sea_geojson, f)
print(f"  → {os.path.getsize(OUTPUT_CATCH)/1e6:.1f} MB  ({OUTPUT_CATCH})")
print("Done.")

## Export tuna_1deg and tuna_1deg 1975

In [ ]:
tuna_1deg_soi_grouped.to_csv("../datasets/export/bluefin-tuna_1deg_soi_grouped.csv", index=False)
tuna_1deg_1975.to_csv("../datasets/export/bluefin-tuna_1deg_soi_grouped_1975.csv", index=False)

## NOAA Import/Export

https://www.fisheries.noaa.gov/foss/f?p=215:2:14285563639484:::::

In [ ]:
fish_import_export = pd.read_csv("datasets/noaa_import_export_tunas.csv")
#column names are in first row, so set header to 0 and skip the first row
fish_import_export.columns = fish_import_export.iloc[0]
fish_import_export = fish_import_export[1:]
fish_import_export.head()


In [ ]:
#year to int
fish_import_export['Year'] = fish_import_export['Year'].astype(int)
# value and volume have commas, so remove commas and convert to int
fish_import_export['Value (USD)'] = fish_import_export['Value (USD)'].str.replace(',', '').astype(int)
fish_import_export['Volume (kg)'] = fish_import_export['Volume (kg)'].str.replace(',', '').astype(int)
#drop calculated duty and edible code
fish_import_export = fish_import_export.drop(columns=['Calculated Duty (USD)', 'Edible code'])




In [ ]:
fish_import_export.dtypes
fish_import_export.head()


In [ ]:
#get rid of:
    #anything that includes the word EVISCERATED
    #OTHER PREPARATIONS
keywords_to_remove = ['EVISCERATED', 'OTHER PREPARATIONS'] #use these keywords to filter
fish_import_export = fish_import_export[~fish_import_export["Product Name"].str.contains('|'.join(keywords_to_remove))]

#All Product Name containing TUNA BLUEFIN become TUNA BLUEFIN 
fish_import_export["Product Name"] = fish_import_export["Product Name"].str.replace(r'TUNA BLUEFIN.*', 'TUNA BLUEFIN', regex=True)
#same for YELLOWFIN and BIGEYE
fish_import_export["Product Name"] = fish_import_export["Product Name"].str.replace(r'TUNA YELLOWFIN.*', 'TUNA YELLOWFIN', regex=True)
fish_import_export["Product Name"] = fish_import_export["Product Name"].str.replace(r'TUNA BIGEYE.*', 'TUNA BIGEYE', regex=True)


fish_import_export["Product Name"].value_counts()


In [ ]:
#also prepare bluefin import only subsection
fish_import_bluefin = fish_import_export[fish_import_export["Product Name"].str.contains("TUNA BLUEFIN")]
fish_import_bluefin = fish_import_bluefin[fish_import_bluefin["Source"] == "IMP"]
fish_import_bluefin.head()

### Export NOAA imp/exp

In [ ]:
fish_import_export.to_csv("datasets/export/noaa_import_export_tunas_combined_cleaned.csv", index=False)
fish_import_bluefin.to_csv("datasets/export/noaa_import_tunas_bluefin.csv", index=False)

# Global Tuna Aquaculture
From UN FAO 

link: https://www.fao.org/fishery/en/collection/global_production?lang=en

big dataset, filtering down for only bluefin tuna

Columns needed:

- PRODUCTION_SOURCE_DET.CODE
    - CAPTURE is wild caught
    - MARINE is marine aquaculture
- PERIOD (year)
- MEASURE
    - Q_tlw is tonnes - live weight
- AREA.CODE
    - gives production area. could be useful?


In [ ]:
#import helper datasets 
country_codes = pd.read_csv("../datasets/GlobalProduction/CL_FI_COUNTRY_GROUPS.csv")
area_codes = pd.read_csv("../datasets/GlobalProduction/CL_FI_WATERAREA_GROUPS.csv")
country_codes.head() #take UN_Code, ISO3_Code, and Name_En columns for assinging country names
area_codes.head() #take Code and Name_En cols


#import dataset
global_production = pd.read_csv("../datasets/GlobalProduction/Global_production_quantity.csv")
global_production.head()

In [ ]:
#bluefin tuna specific dataset:
gp_bluefin = global_production[global_production['SPECIES.ALPHA_3_CODE'].isin(['SBF', 'BFT', 'PBF'])]
gp_bluefin.head()


In [ ]:
def assign_names(df, country_codes, area_codes):
    """
    Assigns country and area names to gp_bluefin dataset.
    
    Parameters:
    - df: gp_bluefin dataframe
    - country_codes: dataframe with UN_Code and Name_En columns
    - area_codes: dataframe with Code and Name_En columns
    
    Returns:
    - df with added 'country_name' and 'area_name' columns
    """
    # Create lookup dictionaries
    country_lookup = dict(zip(country_codes['UN_Code'], country_codes['Name_En']))
    area_lookup = dict(zip(area_codes['Code'], area_codes['Name_En']))
    
    # Map names using the lookups
    df['country_name'] = df['COUNTRY.UN_CODE'].map(country_lookup)
    df['area_name'] = df['AREA.CODE'].map(area_lookup)
    
    return df

# Apply the function
gp_bluefin = assign_names(gp_bluefin, country_codes, area_codes)

In [ ]:
#drop cols: COUNTRY.UN_CODE, AREA.CODE, MEASURE, STATUS
gp_bluefin = gp_bluefin.drop(columns=['COUNTRY.UN_CODE', 'AREA.CODE', 'MEASURE', 'STATUS'])
#rename SPECIES.ALPHA_3_CODE to species, PRODUCTION_SOURCE_DET.CODE to production_type, PERIOD to year, VALUE to measurement_value
gp_bluefin = gp_bluefin.rename(columns={
    'SPECIES.ALPHA_3_CODE': 'species',
    'PRODUCTION_SOURCE_DET.CODE': 'production_type',
    'PERIOD': 'year',
    'VALUE': 'measurement_value'
})


In [ ]:
gp_bluefin.head()

In [ ]:
#groupy by species, and then year, and then production type
gp_bluefin_grouped = gp_bluefin.groupby(['year', 'production_type'])['measurement_value'].sum().reset_index()
gp_bluefin_grouped.head()

In [ ]:
#graph bluefin tuna production over time by production type using gp_bluefin_grouped stacked barplot with year on x measurement on y


# Create pivot table for stacked barplot
gp_bluefin_pivot = gp_bluefin_grouped.pivot(index='year', columns='production_type', values='measurement_value').fillna(0)

# Create stacked barplot
gp_bluefin_pivot.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.xlabel('Year')
plt.ylabel('Production (tonnes)')
plt.title('Global Bluefin Tuna Production Over Time by Production Type')
plt.legend(title='Production Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### export grouped data - only aquaculture/wild capture by year

In [ ]:
#export cleaned and grouped bluefin
gp_bluefin_grouped.to_csv("../datasets/export/bluefin_aquaculture.csv", index=False)

# ICCAT Tuna Aquaculture Locations

https://www.iccat.int/en/Ffb.asp has a list of all ICCAT registered Farming Facilities for BFT. Trying to scrape it.

In [ ]:
import subprocess
result = subprocess.run(['playwright', 'install', 'chromium'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO

async def scrape_iccat_all_pages():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        all_dfs = []
        base_url = 'https://www.iccat.int/en/ffbres.asp?s0=All&s1=1&s2=&s3=&s4=&s5=&s6=&s7='
        
        # Establish session first
        await page.goto('https://www.iccat.int/en/Ffb.asp', wait_until='networkidle')
        
        for offset in range(0, 75, 10):
            url = base_url + f'&offset={offset}'
            print(f"Fetching offset {offset}...")
            
            await page.goto(url, wait_until='networkidle')
            html = await page.content()
            
            soup = BeautifulSoup(html, 'html.parser')
            table = soup.find('table')
            
            if not table:
                print(f"  No table at offset {offset}, stopping")
                break
            
            df = pd.read_html(StringIO(str(table)))[0]
            df = df.dropna(how='all')
            df = df[~df.iloc[:, 0].astype(str).str.contains('Records|ICCAT Serial', na=False)]
            
            print(f"  Got {len(df)} rows")
            all_dfs.append(df)
        
        await browser.close()

        df_all = pd.concat(all_dfs, ignore_index=True)
        # Rename columns using the first row if they're numeric
        if df_all.columns.dtype == 'int64':
            df_all.columns = df_all.iloc[0]
            df_all = df_all.iloc[1:]
        
        return df_all

df = await scrape_iccat_all_pages()
print(f"\n✅ Total: {len(df)} rows")
print(df.head())
df.to_csv('../datasets/aquaculture/iccat_fish_farms.csv', index=False)

once locations are saved to CSV, need to add column names, get rid of unneccesary data, and split locations into appropriate sections

In [ ]:
#import CSV and assign column names
med_tuna_farms = pd.read_csv('../datasets/aquaculture/iccat_fish_farms.csv')

In [ ]:
COLNAMES_AQUACULTURE = [
    "ICCAT Serial Number",
    "Country",
    "Reg.Number","Name of FFB",
    "Owner", #drop
    "Owner Address", #drop
    "Operator", #drop
    "Operator Address", #drop
    "Location", 
    "Capacity (t)"
]

med_tuna_farms.columns = COLNAMES_AQUACULTURE
med_tuna_farms.head()


In [ ]:
#drop columns: Owner, Owner Address, Operator, Operator Address
med_tuna_farms = med_tuna_farms.drop(columns=['Owner', 'Owner Address', 'Operator', 'Operator Address'])


#Names to English
    #Get rid of "EU-", if present
    # Maropc -> Morocco
    # España -> Spain 
    #Tunisie -> Tunisia
med_tuna_farms["Country"] = med_tuna_farms["Country"].str.replace(r'^EU-', '', regex=True)
med_tuna_farms["Country"] = med_tuna_farms["Country"].replace({
    'Maroc': 'Morocco',
    'España': 'Spain',
    'Tunisie': 'Tunisia',
    'Algerie': 'Algeria',
    'Türkiye': 'Turkey',
})
med_tuna_farms.head()


In [ ]:
#export import cleaned
med_tuna_farms.to_csv('../datasets/aquaculture/iccat_fish_farms_dropped.csv', index=False)
med_tf_dropped = pd.read_csv('../datasets/aquaculture/iccat_fish_farms_dropped.csv')
med_tf_dropped.head()

In [ ]:
# Location splitter: parse mixed coordinate formats into decimal latitude/longitude lists
import json
import re

import pandas as pd
from pyproj import Transformer

utm33_to_wgs84 = Transformer.from_crs("EPSG:32633", "EPSG:4326", always_xy=True)


def normalize_location_text(value):
    if pd.isna(value):
        return ""

    text = str(value).translate(
        str.maketrans(
            {
                "º": "°",
                "˚": "°",
                "’": "'",
                "‘": "'",
                "´": "'",
                "`": "'",
                "′": "'",
                "”": '"',
                "“": '"',
                "″": '"',
                "\u00a0": " ",
            }
        )
    )
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"(?i)(?<=\d)\s*O\b", "W", text)  # O = oeste/ouest/west in some rows.
    text = re.sub(r"(?i)\bLONGITUD\b|\bLONGITUDE\b|\bLONG\b", "LONG", text)
    text = re.sub(r"(?i)\bLATITUD\b|\bLATITUDE\b|\bLAT\b", "LAT", text)
    text = re.sub(r"(\d+),\s*(\d+)\s*(?=[\'\"°]|\.?\s*[NSEWO]\b)", r"\1.\2", text)
    text = re.sub(r"(\d+)\s*\.\s*(\d+)\s*(?=[\'\"°]|\s*[NSEWO]\b)", r"\1.\2", text)
    text = re.sub(r"(°\s*\d+(?:\.\d+)?)\s*°\s*([NSEWO])", r"\1'\2", text, flags=re.I)
    return text


def parse_number(value):
    if value is None:
        return 0.0

    cleaned = re.sub(r"[^0-9.+-]", "", str(value).strip().replace(",", "."))
    if cleaned in {"", ".", "+", "-"}:
        return 0.0
    return float(cleaned)


def signed(value, direction):
    direction = (direction or "").upper().replace("O", "W")
    return -value if direction in {"S", "W"} else value


def dms_to_decimal(degrees, minutes=None, seconds=None, direction=None):
    degrees = parse_number(degrees)
    minutes = parse_number(minutes)
    seconds = parse_number(seconds)
    value = abs(degrees) + minutes / 60 + seconds / 3600
    if degrees < 0:
        value = -value
    return round(signed(value, direction), 6)


def decimal_to_signed(value, direction=None):
    return round(signed(abs(parse_number(value)), direction), 6)


def axis_for(direction):
    direction = direction.upper().replace("O", "W")
    return "lat" if direction in {"N", "S"} else "lon"


def add_token(tokens, used_spans, match, axis, value):
    span = match.span()
    if any(not (span[1] <= start or span[0] >= end) for start, end in used_spans):
        return
    tokens.append({"span": span, "axis": axis, "value": value})
    used_spans.append(span)


def collect_directional_tokens(text, prefix_only=False):
    tokens = []
    used_spans = []

    dotted = re.compile(
        r"(?<!\d)(?P<deg>\d{1,3})\.(?P<min>\d{1,2})\.(?P<sec>\d{1,2}(?:\.\d+)?)\s*\(?\s*(?P<dir>[NSEWO])\s*\)?",
        re.I,
    )
    dms_after = re.compile(
        r"(?<![\d.])(?P<deg>\d{1,4})\s*°\s*(?P<min>\d{1,2}(?:\.\d+)?)?\s*(?:[\']\s*(?P<sec>\d{1,2}(?:\.\d+)?)?\s*(?:[\'\"]{0,2})?)?\s*\(?\s*(?P<dir>[NSEWO])\s*\)?",
        re.I,
    )
    dms_before = re.compile(
        r"(?<![A-Z0-9\'\"])(?P<dir>[NSEWO])\s*(?P<deg>\d{1,4})\s*°\s*(?P<min>\d{1,2}(?:\.\d+)?)?\s*(?:[\']\s*(?P<sec>\d{1,2}(?:\.\d+)?)?\s*(?:[\'\"]{0,2})?)?",
        re.I,
    )
    decimal_after = re.compile(r"(?<![\d.])(?P<val>\d{1,3}\.\d+)\s*\(?\s*(?P<dir>[NSEWO])\s*\)?", re.I)

    patterns = [(dms_before, "dms")] if prefix_only else [
        (dotted, "dms"),
        (dms_after, "dms"),
        (dms_before, "dms"),
        (decimal_after, "decimal"),
    ]

    for pattern, kind in patterns:
        for match in pattern.finditer(text):
            direction = match.group("dir").upper().replace("O", "W")
            value = (
                decimal_to_signed(match.group("val"), direction)
                if kind == "decimal"
                else dms_to_decimal(match.group("deg"), match.group("min"), match.group("sec"), direction)
            )
            add_token(tokens, used_spans, match, axis_for(direction), value)

    return sorted(tokens, key=lambda token: token["span"][0])


def pair_tokens(tokens):
    coords = []
    pending = {}
    for token in tokens:
        pending[token["axis"]] = token["value"]
        if "lat" in pending and "lon" in pending:
            coords.append({"lat": pending["lat"], "lon": pending["lon"]})
            pending = {}
    return coords


def infer_lon_sign(country, text, lon):
    if re.search(r"\bW\b|OUEST|ATLANT", text.upper()) or str(country) in {"EU-España", "EU-Portugal", "Maroc"}:
        return -abs(lon)
    return abs(lon)


def parse_utm33_pairs(text):
    if not re.match(r"^\s*N\s+\d{6,}", text, re.I):
        return []

    match = re.match(r"^\s*N\s+(?P<northings>.*?)\s+E\s+(?P<eastings>.*)$", text, re.I)
    if not match or "°" in match.group("northings"):
        return []

    northings = [float(num) for num in re.findall(r"\d+(?:\.\d+)?", match.group("northings"))]
    eastings = [float(num) for num in re.findall(r"\d+(?:\.\d+)?", match.group("eastings"))]
    coords = []
    for northing, easting in zip(northings, eastings):
        lon, lat = utm33_to_wgs84.transform(easting, northing)
        coords.append({"lat": round(lat, 6), "lon": round(lon, 6)})
    return coords


def parse_bare_decimal_pairs(text, country):
    if not re.search(r"(?i)latitude\s*-\s*longitude|\d+\.\d+\s+\d+\.\d+", text):
        return []

    numbers = [float(num) for num in re.findall(r"(?<!\d)(\d{1,3}\.\d+)(?!\d)", text)]
    coords = []
    for lat, lon in zip(numbers[0::2], numbers[1::2]):
        if -90 <= lat <= 90 and -180 <= lon <= 180:
            coords.append({"lat": round(lat, 6), "lon": round(infer_lon_sign(country, text, lon), 6)})
    return coords


def parse_bare_dms_pairs(text, country):
    bare_dms = re.compile(
        r"(?<![\d.])(?P<deg>\d{1,4})\s*°\s*(?P<min>\d{1,2}(?:\.\d+)?)?\s*(?:[\']\s*(?P<sec>\d{1,2}(?:\.\d+)?)?\s*(?:[\'\"]{0,2})?)?"
    )
    values = []
    for match in bare_dms.finditer(text):
        before = text[max(0, match.start() - 2) : match.start()].upper()
        after = text[match.end() : match.end() + 4].upper()
        if re.search(r"[NSEWO]\s*$", before) or re.search(r"^\s*[NSEWO]\b", after):
            continue
        values.append(dms_to_decimal(match.group("deg"), match.group("min"), match.group("sec")))

    coords = []
    for lat, lon in zip(values[0::2], values[1::2]):
        if 0 <= lat <= 90 and 0 <= abs(lon) <= 180:
            coords.append({"lat": round(lat, 6), "lon": round(infer_lon_sign(country, text, lon), 6)})
    return coords


def valid_coord(coord):
    return -90 <= coord["lat"] <= 90 and -180 <= coord["lon"] <= 180


def parse_location(location, country=None):
    original = "" if pd.isna(location) else str(location).strip()
    if not original:
        return []

    text = normalize_location_text(original)
    prefix_coords = pair_tokens(collect_directional_tokens(text, prefix_only=True))
    if prefix_coords and len(collect_directional_tokens(text, prefix_only=True)) >= 2:
        coords = prefix_coords
    else:
        coords = (
            parse_utm33_pairs(original)
            or pair_tokens(collect_directional_tokens(text))
            or parse_bare_decimal_pairs(text, country)
            or parse_bare_dms_pairs(text, country)
        )

    coords = [coord for coord in coords if valid_coord(coord)]
    return coords if coords else [original]


def parsed_as_coordinates(value):
    return bool(value) and isinstance(value[0], dict)


med_tf_dropped = pd.read_csv("../datasets/aquaculture/iccat_fish_farms_dropped.csv")
med_tf_dropped["cleaned_locations"] = med_tf_dropped.apply(
    lambda row: parse_location(row["Location"], row["Country"]), axis=1
)

export_df = med_tf_dropped.copy()
export_df["cleaned_locations"] = export_df["cleaned_locations"].apply(json.dumps)
export_path = "../datasets/aquaculture/iccat_fish_farms_locations_cleaned.csv"
export_df.to_csv(export_path, index=False)

parsed_mask = med_tf_dropped["cleaned_locations"].apply(parsed_as_coordinates)
fallback_rows = med_tf_dropped.loc[~parsed_mask, ["ICCAT Serial Number", "Country", "Name of FFB", "Location", "cleaned_locations"]]

print(f"Exported {len(export_df)} rows to {export_path}")
print(f"Parsed coordinate rows: {parsed_mask.sum()}")
print(f"Fallback rows: {(~parsed_mask).sum()}")
print("\nFallback rows:")
display(fallback_rows)

# These rows contain coordinates in the source text but need manual review because the ICCAT text is truncated or malformed.
suspicious = med_tf_dropped[
    parsed_mask
    & med_tf_dropped["cleaned_locations"].apply(
        lambda coords: any(abs(coord["lon"]) > 30 or coord["lat"] < 25 for coord in coords if isinstance(coord, dict))
    )
][["ICCAT Serial Number", "Country", "Name of FFB", "Location", "cleaned_locations"]]

print("\nSuspicious parsed rows to review:")
display(suspicious)

print("\nRepresentative parsed rows:")
display(
    med_tf_dropped.loc[
        med_tf_dropped["ICCAT Serial Number"].isin(
            ["AT001DZA00001", "ATEU1HRV00011", "ATEU1GRC00002", "ATEU1MLT00001", "ATEU1PRT00003"]
        ),
        ["ICCAT Serial Number", "Country", "Location", "cleaned_locations"],
    ]
)

In [35]:
#drop some ICCAT from CSV before creating geojson (mainly the ones on land)

# import csv
med_tf_cleaned = pd.read_csv("../datasets/aquaculture/iccat_fish_farms_locations_cleaned.csv")

#list of ICCAT numbers to drop:
iccat_to_drop = ["AT001TUR00014","ATEU1ESP00002","ATEU1ESP00004"]

med_tf_cleaned = med_tf_cleaned[~med_tf_cleaned["ICCAT Serial Number"].isin(iccat_to_drop)]

#to csv
med_tf_cleaned.to_csv("../datasets/aquaculture/iccat_fish_farms_locations_cleaned_no_land.csv", index=False)


In [36]:
# ICCAT farms → centroid GeoJSON (for map prototypes; see datasets/export/geojson_export)
from pathlib import Path
import json

import pandas as pd

ICCAT_CSV = Path("../datasets/aquaculture/iccat_fish_farms_locations_cleaned_no_land.csv")
GEOJSON_OUT = Path("../datasets/export/geojson_export/iccat_fish_farms_centroids.geojson")


def centroid_from_cleaned_locations(serialized):
    """Return (lon, lat, vertex_count) or None."""
    if pd.isna(serialized) or str(serialized).strip() == "":
        return None
    try:
        data = json.loads(serialized)
    except json.JSONDecodeError:
        return None
    pts = []
    for item in data:
        if not isinstance(item, dict):
            continue
        try:
            lat = float(item["lat"])
            lon = float(item["lon"])
        except (KeyError, TypeError, ValueError):
            continue
        if -90 <= lat <= 90 and -180 <= lon <= 180:
            pts.append((lon, lat))
    if not pts:
        return None
    mlon = sum(p[0] for p in pts) / len(pts)
    mlat = sum(p[1] for p in pts) / len(pts)
    return mlon, mlat, len(pts)


def properties_for_geojson(row):
    cap = row.get("Capacity (t)")
    if pd.isna(cap):
        capacity = None
    else:
        capacity = float(cap)
    return {
        "iccat_serial": row.get("ICCAT Serial Number"),
        "country": row.get("Country"),
        "reg_number": row.get("Reg.Number"),
        "name": row.get("Name of FFB"),
        "capacity_t": capacity,
    }


farms = pd.read_csv(ICCAT_CSV)
features = []
skipped_serials = []

for _, row in farms.iterrows():
    result = centroid_from_cleaned_locations(row["cleaned_locations"])
    if result is None:
        skipped_serials.append(row.get("ICCAT Serial Number"))
        continue
    mlon, mlat, n = result
    props = properties_for_geojson(row)
    props["vertex_count"] = n
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [mlon, mlat]},
        "properties": props,
    })

collection = {"type": "FeatureCollection", "features": features}
GEOJSON_OUT.parent.mkdir(parents=True, exist_ok=True)
GEOJSON_OUT.write_text(json.dumps(collection, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Wrote {len(features)} features → {GEOJSON_OUT.resolve()}")
print(f"Skipped rows (no usable lat/lon in cleaned_locations): {len(skipped_serials)}")
if skipped_serials:
    tail = " …" if len(skipped_serials) > 12 else ""
    print(skipped_serials[:12], tail)


Wrote 52 features → /Users/nichosmolnar/Documents/Spring_2026_Parsons/Major_Studio_2/thesis/datasets/export/geojson_export/iccat_fish_farms_centroids.geojson
Skipped rows (no usable lat/lon in cleaned_locations): 19
['ATEU1HRV00012', 'ATEU1HRV00008', 'ATEU1GRC00001', 'ATEU1ITA00005', 'ATEU1ITA00007', 'ATEU1ITA00016', 'ATEU1ITA00017', 'ATEU1ITA00019', 'ATEU1ITA00022', 'ATEU1ITA00023', 'ATEU1ITA00024', 'ATEU1ITA00025']  …
